# EC classifier selection and training

Tune the EC classifier with Optuna, track cross-validation in W&B and evaluate the final network. [Trials and fold scores](../../results/task2/)


In [ ]:
import sys
sys.path.append('..')

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
from src.preprocessing import *
from src.utils import *
from sklearn.metrics import balanced_accuracy_score
import torch
from torch.utils.data import Subset
from torch import nn
from src.models import *
import wandb
from src.crossval import *
import torch.optim as optim
from src.train import *


In [ ]:
train_path = Path('../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp2000_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp2000_exp_train85_dim128.h5')
test_path = Path('../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp2000_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp2000_exp_test85_dim128.h5')
train_set = create_dataset_from_h5(train_path, train_path.stem, unsqueeze=False, multiclass=False)
test_set = create_dataset_from_h5(test_path, test_path.stem, unsqueeze=False, multiclass=False)

In [ ]:
def objective(trial):
    cap = 1500 # trial.suggest_categorical('dataset', [1500, 1808, 2000, 2500])
    train_path = Path(f'../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp{cap}_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp{cap}_exp_train85_dim128.h5')
    test_path = Path(f'../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp{cap}_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp{cap}_exp_test85_dim128.h5')
    train_set = create_dataset_from_h5(train_path, train_path.stem, unsqueeze=False, multiclass=False)
    # Model architecture
    n_hidden_layers = 1
    input_dim = 128
    output_dim = 7
    possible_dims = [16, 32, 64, 128]
    dims = [input_dim, 32, 7]
    
    # for i in range(n_hidden_layers):
    #     temp_layer_size = trial.suggest_categorical(f'hidden_layer_{i}', possible_dims)
    #     dims.append(temp_layer_size)

    # dims.append(output_dim)
    trial.set_user_attr('dims', dims)

    # Dropouts
    dropout_0 = trial.suggest_float(f'dropout_0', 0.3, 0.33)

    #n_dropouts = 2
    dropouts = [dropout_0]
    # for i in range(n_dropouts):
    #     dropouts.append(trial.suggest_float(f'dropout_{i}', 0.5, 0.75))

    activation = nn.GELU() # trial.suggest_categorical('activation', ['LeakyReLU', 'GELU'])
    # if activation == 'ReLU':
    #     activation = nn.LeakyReLU()
    # else:
    #     activation = nn.GELU()
    lr = trial.suggest_float('lr', 0.014, 0.015)
    weight_decay = trial.suggest_float('weight_decay', 0.035, 0.04)
    optimizer = optim.AdamW #trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])
    # if optimizer == 'Adam':
    #     optimizer = optim.Adam
    # else:
    #     optimizer = optim.AdamW
    class_dist_list = get_class_distribution_list(train_set, relative=False)
    n = sum(class_dist_list)
    class_weights = torch.tensor([n/(7 * p) for p in class_dist_list], dtype=torch.float32)
    label_smoothing = trial.suggest_float('label_smoothing', 0, 0.05)
    loss = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    batch_size = 16 #trial.suggest_categorical('batch_size', [16, 32, 64])
    epochs = 200

    model = Task2Model(dims=dims, dropouts=dropouts, activation=activation)

    train_losses, val_losses, train_f1, val_f1, train_mcc, val_mcc, cv_results = cross_validate_wandb(model_in=model, dataset=train_set, weight_decay=weight_decay, optimizer_fn=optimizer, loss_fn=loss, n_splits=5, batch_size=batch_size, lr=lr, epochs=epochs, patience=12)
    wandb.init(project=f'task2_model_selection_{trial.study.study_name}', name=f'cv_nn_v{trial.number}')
    cv_table = wandb.Table(dataframe=cv_results)
    f1_curve = visualize_cv_spaghetti(train_f1, val_f1, 'macro_f1', False)
    loss_curve = visualize_cv_spaghetti(train_losses, val_losses, 'loss', False)
    mcc_curve = visualize_cv_spaghetti(train_mcc, val_mcc, 'loss', False)

    overfitting_metric_loss, overfitting_metric_f1 = calculate_gap_means(train_losses, val_losses, train_f1, val_f1) 
    overfitting_metric = 0.7 * overfitting_metric_f1 + 0.3 * overfitting_metric_loss

    wandb.log({'dataset': train_path.stem, 'dims': dims, 'dropouts': dropouts, 'lr': lr, 'weight_decay': weight_decay, 'optimizer': optimizer, 'loss': loss, 'label_smoothing': label_smoothing, 'epochs': epochs, 'batch_size': batch_size, 'cv_loss_curves': wandb.Image(loss_curve), 'cv_f1_curves':  wandb.Image(f1_curve), 'cv_mcc_curves': wandb.Image(mcc_curve), 'cv_balanced_acc': np.mean(cv_results['balanced_accuracy']), 'cv_mcc': np.mean(cv_results['mcc']), 'cv_macro_f1': np.mean(cv_results['macro_f1']), 'overfitting_metric_loss': overfitting_metric_loss, 'overfitting_metric_f1': overfitting_metric_f1, 'cv_results': cv_table})
    plt.close()
    wandb.finish()

    return np.mean(cv_results['macro_f1']), overfitting_metric

In [ ]:
study = optuna.create_study(
    storage="sqlite:///db.sqlite3",
    study_name="finest_search_one_layer_LeakyReLU_v3_no_label_smoothing",
    directions=["maximize", 'minimize'],
    load_if_exists=True
)
study.optimize(objective, n_trials=50)

In [ ]:
model = Task2Model(dims=[128, 256, 7], dropouts=[0.11764897516223502], activation=nn.LeakyReLU())
lr = 0.010407922166686873
weight_decay = 0.03424566834898443
label_smoothing = 0.066
weights_alpha = 0.496
weight_power = 0.483

class_dist_list = get_class_distribution_list(train_set, relative=False)
counts = torch.tensor(class_dist_list, dtype=torch.float32)

w = (counts.sum() / (7.0 * torch.clamp(counts, min=1.0))).pow(weight_power)
w = (w / w.mean()).to(device)

criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)

train_loss, val_loss, train_f1, val_f1, train_mcc, val_mcc, results = cross_validate_wandb(model_in=model, dataset=train_set, optimizer_fn=optim.AdamW, weight_decay=weight_decay, loss_fn=criterion, n_splits=5, batch_size=16, lr=lr, epochs=200, patience=12)
#trained_model, loss_curve, f1_curve = train_model(model=model, model_name=f'task2_test_model_{train_set.name}_one_layer', dataset=train_set, optimizer=optimizer, criterion=criterion, epochs=100, early_stopping=True, patience=10, show_training_curves=True, batch_size=16)

In [ ]:
np.mean(results['macro_f1'])

In [ ]:
visualize_cv_spaghetti(train_f1, val_f1, metric='Macro F1')

In [ ]:
model = Task2Model(dims=[128, 256, 7], dropouts=[0.11764897516223502], activation=nn.LeakyReLU())
lr = 0.010407922166686873
weight_decay = 0.03424566834898443
label_smoothing = 0.066
weights_alpha = 0.496
weight_power = 0.483
optimizer = optim.AdamW(params=model.parameters(), weight_decay=weight_decay)

class_dist_list = get_class_distribution_list(train_set, relative=False)
counts = torch.tensor(class_dist_list, dtype=torch.float32)

w = (counts.sum() / (7.0 * torch.clamp(counts, min=1.0))).pow(weight_power)
w = (w / w.mean()).to(device)

criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)

train_model(model=model, model_name=f'task_2_first_model_{train_set.name}', dataset=train_set, optimizer=optimizer, criterion=criterion, epochs=200, early_stopping=True, patience=12, batch_size=16, sampler_alpha=weights_alpha)

In [ ]:
# Portfolio adaptation for the precomputed 128-dimensional test embeddings.
from torch.utils.data import DataLoader

model = build_model(Path('../../configs/task2.yaml'))
weights = torch.load('../../checkpoints/task2.pth', map_location='cpu', weights_only=True)
model.load_state_dict(weights)
balanced_acc, macro_f1, mcc, confusion = get_cv_metrics(model, DataLoader(test_set, batch_size=16))
print(balanced_acc, macro_f1, mcc)
print(confusion.compute())

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
cm = np.array([[261,  13,   8,  10,   2,   3,   3],
        [  8, 239,  23,  15,   8,   4,   3],
        [ 13,  26, 238,  17,   1,   3,   2],
        [ 14,  11,   8,  70,  10,   1,   1],
        [  5,   3,  10,  11,  45,   1,   0],
        [  2,   3,   2,   2,   3,  43,   0],
        [  2,   2,   2,   0,   0,   2,  25]])

# 2. Calculate Percentages (Row-wise normalization)
# We divide each value by the sum of its row (True Class total)
cm_perc = cm.astype('float') / cm.sum()# 3. Initialize Display
disp = ConfusionMatrixDisplay(confusion_matrix=cm_perc, display_labels=range(1,8))

# 4. Create Plot
fig, ax = plt.subplots(figsize=(10, 10)) # Increased figure size for visibility

# Plot with custom settings
disp.plot(
    cmap='Blues', 
    ax=ax, 
    colorbar=False, 
    values_format='.1%',     # Removes the bar on the right
    text_kw={'size': 20} # Makes the numbers inside the matrix bigger
)

# 5. Customize Axis Labels and Ticks
# Set labels AFTER plotting to ensure they persist
ax.set_xlabel('Predicted EC class', fontsize=18)
ax.set_ylabel('Actual EC class', fontsize=18)

# Make the axis numbers (1-7) bigger
ax.tick_params(axis='x', labelsize=16)
ax.tick_params(axis='y', labelsize=16)

# Title
plt.title("Confusion Matrix Task 2", fontsize=20)
plt.tight_layout()
plt.show()